In [7]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_percentage_error

In [8]:
#Log Likelyhood: hoch, AIC: niedrig

def ar_arima(rge,df_train, df_test, df, header, start_date, end_date):

    df_train.index = pd.to_datetime(df_train.index)
    df_train = df_train.asfreq('D')
    
    for i in range(rge):
        p, d, q = i, 1, 0

        arima = SARIMAX(df_train[header], order=(p, d, q), freq='D')
        result = arima.fit()
        print(result.summary())
        # Update des DataFrames
        df['differences'] = df_train['differences']
        model = SARIMAX(df[header], order=(p, d, q), freq='D')
        result_new = model.filter(result.params)

        # Vorhersage
        predicted = result_new.predict(start=start_date)
        tmp = pd.DataFrame(predicted)
        tmp['crossings'] = tmp['predicted_mean']
        tmp.index = tmp.index.date

        # Mergen der Daten
        df_r = pd.concat([df_train, tmp['crossings']], axis=1)

        #df_r = df_train.merge(tmp, how='left', left_index=True, right_index=True)

        df_r[header] = df_r[header].fillna(df_r['crossings'])
        df_r = pd.DataFrame(df_r)
        df_r.index = pd.to_datetime(df_r.index)

        # Plotten
        plt.title('AR, Ordnung = {:.0f}'.format(i))
        plt.plot(df_r.loc[start_date:end_date, header],
             color='yellow', label='predicted')
        plt.plot(df.loc[start_date:end_date, header], color='blue', label='actual')
        plt.legend()
        plt.show()

        y_pred_simple = df_test[header].iloc[:-1]
        y_pred_simple = pd.concat([y_pred_simple, df_train[header].iloc[-1:]])
        y_pred_simple = y_pred_simple.sort_index().tolist()

        df_test.loc[:,'intervals'] = (df_test[header] > 10).astype(int).diff().fillna(0).ne(0).cumsum()
        df_test['group'] = df_test[header].apply(lambda x: '≤ 10' if x <= 10 else '> 10')
        df_test.loc[:,'predicted'] = df_r.loc[start_date:end_date, header]
        df_test.loc[:,'simple_predict'] = y_pred_simple

        greater = df_test[df_test['group'] == '> 10']
        mape_arima = []
        mape_simple = []

        for name, group in greater.groupby('intervals'):
            mape_arima.append(mean_absolute_percentage_error(group[header], group['predicted']))
            mape_simple.append(mean_absolute_percentage_error(group[header], group['simple_predict']))
            '''
            if df_test[header].iloc[y] > 10:
                act = df_test[header].iloc[y]
                pred_ar = df_test['predicted'].iloc[y]
                pred_simp = df_test['simple_predict'].iloc[y]
                mape_arima_list[0].append(act)
                mape_arima_list[1].append(pred_ar)
                mape_simple_list[0].append(act)
                mape_simple_list[1].append(pred_simp)
                '''
        #display(mape_arima)
        #display(mape_simple)

        mae_arima = np.mean(np.abs(df_test[header] - predicted))
        mse_arima = np.mean(np.square(df_test[header] - predicted))
        mape_arima = sum(mape_arima)/len(mape_arima)
        mae_simple = np.mean(np.abs(df_test[header] - y_pred_simple))
        mse_simple = np.mean(np.square(df_test[header] - y_pred_simple))
        mape_simple = sum(mape_simple)/len(mape_simple)

        print("mean absolute error (mae): ARIMA: {:.4f}, simple: {:.4f}".format(mae_arima, mae_simple))
        print("mean squared error (mse): ARIMA: {:.4f}, simple: {:.4f}".format(mse_arima, mse_simple))
        print("mean absolute percentage error (mape): ARIMA: {:.4f}, simple: {:.4f}".format(mape_arima, mape_simple))

In [ ]:
def mape_intervals(df, header, boundary, df_predicted):
    df.loc[:,'intervals'] = (df[header] > boundary).astype(int).diff().fillna(0).ne(0).cumsum()
    df.loc[:,'group'] = df[header].apply(lambda x: '≤ {:.}'.format(boundary) if x <= boundary else '> {:.}'.format(boundary))
    df.loc[:,'predicted'] = df_predicted

    greater = df[df['group'] == '> {:.}'.format(boundary)]
    mape_list_greater = []
    mape_list_smaller = []

    for name, group in greater.groupby('intervals'):
        mape_list_greater.append(mean_absolute_percentage_error(group[header], group['predicted']))
        mape_list_smaller.append(mean_absolute_percentage_error(group[header], group['predicted']))


    return [mape_list_greater, mape_list_smaller]